# Challenge 2: Programming a RAG system in BigQuery

## Step 1: Set up dependencies

### Libs

In [1]:
!pip install google-cloud-bigquery db-dtypes google-genai

### Imports

In [2]:
import os
import json
import subprocess

from google.cloud import bigquery
from google import genai

### Config

In [3]:
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "qwiklabs-gcp-02-980c840fda76")
BQ_LOCATION = "US"
VERTEX_LOCATION = "us-east1"
DATASET = "aurora_bay"
CONNECTION_ID = "vertex-ai-connection"
SOURCE_CSV = "gs://labs.roitraining.com/aurora-bay-faqs/aurora-bay-faqs.csv"

EMBEDDING_MODEL = "embedding_model"
GEMINI_MODEL = "gemini-2.5-flash"

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=VERTEX_LOCATION)

### Functions

In [4]:
def run(sql, params=None):
    job_config = bigquery.QueryJobConfig(query_parameters=params or [])
    return bq.query(sql, job_config=job_config).result().to_dataframe()

### Connection

In [5]:
!bq mk --connection --location={BQ_LOCATION} --project_id={PROJECT_ID} \
    --connection_type=CLOUD_RESOURCE {CONNECTION_ID} 2>/dev/null; echo "connection ready"

# Pull the connection's auto-created service account
import json, subprocess
info = subprocess.run(
    ["bq", "show", "--format=prettyjson", "--connection",
     f"{PROJECT_ID}.{BQ_LOCATION}.{CONNECTION_ID}"],
    capture_output=True, text=True).stdout
conn_sa = json.loads(info)["cloudResource"]["serviceAccountId"]
print("Connection SA:", conn_sa)

# Grant Vertex AI User to the connection SA (same project as the model)
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{conn_sa}" \
    --role="roles/aiplatform.user" --condition=None --quiet >/dev/null
print("Granted roles/aiplatform.user")

BigQuery error in mk operation: Already Exists: Connection
projects/565151637439/locations/us/connections/vertex-ai-connection
connection ready
Connection SA: bqcx-565151637439-uwo9@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Updated IAM policy for project [qwiklabs-gcp-02-980c840fda76].
Granted roles/aiplatform.user


## Step 2: Load data

### Create dataset

In [6]:
ds = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds.location = BQ_LOCATION
bq.create_dataset(ds, exists_ok=True)
print(f"Dataset ready: {PROJECT_ID}.{DATASET} ({BQ_LOCATION})")

Dataset ready: qwiklabs-gcp-02-980c840fda76.aurora_bay (US)


### Load dataset

In [7]:
table_id = f"{PROJECT_ID}.{DATASET}.faqs"
load_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
load_job = bq.load_table_from_uri(SOURCE_CSV, table_id, job_config=load_config)
load_job.result()
print(f"Loaded {bq.get_table(table_id).num_rows} rows into {table_id}")

# Preview so you can see the actual column names (likely question / answer)
run(f"SELECT * FROM `{table_id}` LIMIT 5")

Loaded 50 rows into qwiklabs-gcp-02-980c840fda76.aurora_bay.faqs


,string_field_0,string_field_1
0,When was Aurora Bay founded?,Aurora Bay was founded in 1901 by a group of f...
1,What is the population of Aurora Bay?,Aurora Bay has a population of approximately 3...
2,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...
3,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ..."
4,What are the primary industries in Aurora Bay?,The primary industries include commercial fish...


## Step 3: Generate embeddings

### Create embeddings model

In [8]:
run(f"""
CREATE OR REPLACE MODEL
  `{PROJECT_ID}.{DATASET}.{EMBEDDING_MODEL}` REMOTE
WITH CONNECTION `us.{CONNECTION_ID}` OPTIONS (ENDPOINT = 'text-embedding-005')
""")
print("Embedding model created")

Embedding model created


### Generate embeddings

In [9]:
# Embed the question + answer together. Adjust the column names below to match
run(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.faqs_embedded` AS
SELECT *
FROM ML.GENERATE_EMBEDDING(
    MODEL `{PROJECT_ID}.{DATASET}.{EMBEDDING_MODEL}`,
    (
        SELECT
            *,
            CONCAT(string_field_0, ' ', string_field_1) AS content
        FROM `{PROJECT_ID}.{DATASET}.faqs`
    )
)
""")

""


### Test queries

In [10]:
# Sanity check: empty status string means no embedding errors
run(f"""
SELECT COUNT(*) AS data_count,
       COUNTIF(LENGTH(ml_generate_embedding_status) = 0) AS embedded_ok
FROM `{PROJECT_ID}.{DATASET}.faqs_embedded`
""")

,data_count,embedded_ok
0,50,50


In [11]:
# Sanity check: does vector search return sensible nearest neighbors?
test_q = "Where is the Aurora Bay Town Hall located?"

sanity = run(f"""
SELECT
    query.query AS search_text,
    base.string_field_0 AS question,
    base.string_field_1 AS answer,
    distance
FROM VECTOR_SEARCH(
    TABLE `{PROJECT_ID}.{DATASET}.faqs_embedded`,
    'ml_generate_embedding_result',
    (
        SELECT ml_generate_embedding_result, content AS query
        FROM ML.GENERATE_EMBEDDING(
            MODEL `{PROJECT_ID}.{DATASET}.{EMBEDDING_MODEL}`,
            (SELECT @q AS content)
        )
    ),
    top_k => 5
)
ORDER BY distance
""", params=[bigquery.ScalarQueryParameter("q", "STRING", test_q)])

print(f"Query: {test_q}\n")
sanity

Query: Where is the Aurora Bay Town Hall located?



,search_text,question,answer,distance
0,Where is the Aurora Bay Town Hall located?,Where is the Aurora Bay Town Hall located?,The Town Hall is located at 100 Harbor View Ro...,0.384512
1,Where is the Aurora Bay Town Hall located?,Does Aurora Bay have a public library?,Yes. The Aurora Bay Public Library is located ...,0.668609
2,Where is the Aurora Bay Town Hall located?,Does Aurora Bay have a police department?,Yes. The Aurora Bay Police Department is locat...,0.719776
3,Where is the Aurora Bay Town Hall located?,Is there a local hospital in Aurora Bay?,Yes. The Aurora Bay Community Hospital offers ...,0.735782
4,Where is the Aurora Bay Town Hall located?,Who is the current mayor of Aurora Bay?,"The current mayor is Linda Greenwood, elected ...",0.746268


## Step 4: Example RAG app with Gemini

### Vector search

In [12]:
def retrieve(question, top_k=5):
    """Embed the question and nearest-neighbor search the FAQ embeddings."""
    sql = f"""
    SELECT base.content AS content, distance
    FROM VECTOR_SEARCH(
        TABLE `{PROJECT_ID}.{DATASET}.faqs_embedded`,
        'ml_generate_embedding_result',
        (
            SELECT ml_generate_embedding_result, content
            FROM ML.GENERATE_EMBEDDING(
                MODEL `{PROJECT_ID}.{DATASET}.{EMBEDDING_MODEL}`,
                (SELECT @q AS content)
            )
        ),
        top_k => {top_k}
    )
    ORDER BY distance
    """
    params = [bigquery.ScalarQueryParameter("q", "STRING", question)]
    return run(sql, params)

### Generate chat response

In [13]:
def ask(question, top_k=5):
    """Full RAG turn: retrieve context, then have Gemini answer from it."""
    hits = retrieve(question, top_k)
    context = "\n\n".join(hits["content"].tolist())
    prompt = (
        "Answer the question using only the Aurora Bay FAQ context below. "
        "If the answer is not in the context, say you don't have that information.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}"
    )
    resp = genai_client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return resp.text

### Send prompts

In [14]:
print(ask("Where is the Aurora Bay Town Hall located?"))
print(ask("When was Aurora Bay founded?"))
print(ask("What is the Aurora Lights Winter Festival?"))
print(ask("What is the capital of France?"))

The Town Hall is located at 100 Harbor View Road, in the center of Aurora Bay, close to the main harbor.
Aurora Bay was founded in 1901.
It’s Aurora Bay’s annual winter celebration featuring ice carving competitions, northern lights viewing tours, live music, and local food vendors. It typically takes place in late January.
I don't have that information.
